# 🔍 Notebook 07: Vector Database Architecture, FAISS Indexing & Viewpoint-Aware Query Routing
### Capstone: VL-JEPA vs. CLIP for CCTV Surveillance Event Retrieval, Event Summarization & Hallucination Analysis
---
**Objective:** Construct production-grade vector similarity search infrastructure:
- FAISS Index FlatIP vs IndexIVFFlat indexing.
- Index construction across all 5,153 clips.
- Viewpoint-aware query routing (aerial vs ground vs rooftop).
- Latency vs recall tradeoff benchmarking.


## Step 1: Load FAISS Indices for All Datasets
Inspect and verify the FAISS indices created for RLVS, UCF-ARG, TinyVIRAT, UCF-Crime, and Unified.


In [1]:
import sys
from pathlib import Path
import faiss
import json
import time

sys.path.append(str(Path.cwd().parent))
import notebook_utils as nu

retriever = nu.CCTVRetrievalEngine()
print(f"\n=== Loaded Indices Summary ===")
for name, idx in retriever.indices.items():
    print(f"  Index: {name:<12} | Vectors: {idx.ntotal:,} | Metric: Inner Product (Cosine)")


[Retrieval Engine] Loaded 'rlvs' index: 2000 vectors.
[Retrieval Engine] Loaded 'ucf_arg' index: 1410 vectors.
[Retrieval Engine] Loaded 'tinyvirat' index: 743 vectors.
[Retrieval Engine] Loaded 'ucf_crime' index: 1000 vectors.
[Retrieval Engine] Loaded 'unified' index: 5153 vectors.

=== Loaded Indices Summary ===
  Index: rlvs         | Vectors: 2,000 | Metric: Inner Product (Cosine)
  Index: ucf_arg      | Vectors: 1,410 | Metric: Inner Product (Cosine)
  Index: tinyvirat    | Vectors: 743 | Metric: Inner Product (Cosine)
  Index: ucf_crime    | Vectors: 1,000 | Metric: Inner Product (Cosine)
  Index: unified      | Vectors: 5,153 | Metric: Inner Product (Cosine)


## Step 2: Index Latency Benchmarking (FlatIP vs IVFFlat)
Measure search latency across query batches.


In [2]:
import numpy as np

idx_unified = retriever.indices["unified"]
dummy_query = np.random.randn(1, 512).astype(np.float32)
dummy_query /= np.linalg.norm(dummy_query, axis=-1, keepdims=True)

# Benchmark 100 queries
t0 = time.time()
for _ in range(100):
    scores, indices = idx_unified.search(dummy_query, 10)
elapsed = time.time() - t0

latency_ms = (elapsed / 100) * 1000
qps = 100 / elapsed
print(f"FAISS FlatIP Latency: {latency_ms:.3f} ms per query ({qps:.1f} Queries/sec)")


FAISS FlatIP Latency: 0.250 ms per query (3993.9 Queries/sec)


## Step 3: Viewpoint-Aware Query Routing Demonstration
Verify that query filtering respects camera viewpoint constraints (aerial vs ground).


In [3]:
# Aerial test
res_aerial = retriever.query("person jogging", dataset="ucf_arg", viewpoint_filter="aerial", top_k=3)
print("=== Aerial Filtered Query Results ===")
for r in res_aerial:
    vp = r["metadata"].get("viewpoint", "N/A")
    print(f"Rank #{r['rank']} | Score: {r['score']:.4f} | Viewpoint: {vp} | Action: {r['label']}")

# Ground test
res_ground = retriever.query("person jogging", dataset="ucf_arg", viewpoint_filter="ground", top_k=3)
print("\n=== Ground Filtered Query Results ===")
for r in res_ground:
    vp = r["metadata"].get("viewpoint", "N/A")
    print(f"Rank #{r['rank']} | Score: {r['score']:.4f} | Viewpoint: {vp} | Action: {r['label']}")


[Retrieval Engine] Initializing CLIP Text Encoder on cuda...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

=== Aerial Filtered Query Results ===
Rank #1 | Score: 0.2701 | Viewpoint: aerial | Action: walking
Rank #2 | Score: 0.2697 | Viewpoint: aerial | Action: walking
Rank #3 | Score: 0.2661 | Viewpoint: aerial | Action: boxing

=== Ground Filtered Query Results ===
Rank #1 | Score: 0.2666 | Viewpoint: ground | Action: clapping
Rank #2 | Score: 0.2639 | Viewpoint: ground | Action: waving
Rank #3 | Score: 0.2609 | Viewpoint: ground | Action: digging
